# Amaliy mashgʻulot 14. Savol-javob tizimlari (QA) va zich qidiruv (DPR)

14-laboratoriyada siz savol boʻyicha matn parchalarini tartiblagan, ikki oʻlchovli «zich» fazoni, in-batch negatives bilan B×B matritsani va matndan javobni ajratishni koʻrgansiz. Bu yerda hammasini kodda takrorlaymiz:

1. BM25 ni noldan yozamiz va scikit-learn dagi TF-IDF bilan solishtiramiz;
2. qidiruv sifatini Recall@k va MRR metrikalari bilan oʻlchaymiz;
3. numpy da in-batch negatives bilan kichik ikki enkoderli modelni (dual encoder) oʻrgatamiz — DPR xuddi shunday oʻrgatiladi;
4. siyrak va zich qidiruvni savollar boʻyicha solishtiramiz va gibridni sinab koʻramiz;
5. topilgan parchadan javobni ajratib oladigan oddiy «oʻquvchi» (reader) yozamiz.

*«NLP algoritmlari» kursining 14-laboratoriyasiga amaliy mashgʻulot. Kataklarni yuqoridan pastga ishga tushiring: Shift+Enter yoki «Runtime → Run all». Javoblaringizni saqlash uchun «File → Save a copy in Drive» ni bosing.*

## 1. Maʼlumotlar: bilimlar bazasi va savollar

Ochiq savol-javob tizimi (open-domain QA) odatda ikki qismdan iborat: **retriever** (qidiruvchi) katta toʻplamdan bir nechta mos parchani topadi, **reader** (oʻquvchi) ulardan javobni ajratib oladi. Bizning toʻplam — oddiy faktlar yozilgan 18 ta qisqa parcha. Har bir test savoli uchun toʻgʻri parcha va javob maʼlum.

Zich enkoder uchun alohida «savol → parcha» oʻquv juftliklari bor (3-boʻlim). Ular test savollaridan boshqacha tuzilgan, aks holda tekshiruv halol boʻlmasdi.

In [ ]:
# 18 ta qisqa matn parchasi (passage): bu bizning «bilimlar bazamiz»
passages = [
    "Toshkent — Oʻzbekistonning poytaxti va mamlakatdagi eng katta shahar. Unda taxminan uch million odam yashaydi.",
    "Samarqand — Oʻrta Osiyodagi eng qadimiy shaharlardan biri. Shahar markazida uchta madrasali Registon maydoni joylashgan.",
    "Mirzo Ulugʻbek 1420-yillarda Samarqandda rasadxona qurdirgan. Uning yulduzlar jadvalida mingdan ortiq yulduz bor.",
    "Alisher Navoiy — XV asr shoiri va mutafakkiri. U 1441-yilda Hirotda tugʻilgan.",
    "Al-Xorazmiy — IX asrda yashagan, Xorazmdan chiqqan matematik. «Algoritm» soʻzi uning nomidan kelib chiqqan.",
    "Ibn Sino 980-yil atrofida Buxoro yaqinida tugʻilgan. Uning «Tib qonunlari» asari bir necha asr davomida tibbiyotning asosiy darsligi boʻlgan.",
    "Orol dengizi bir vaqtlar dunyodagi eng katta koʻllardan biri edi. Daryolar suvi dalalarni sugʻorishga olingani uchun u juda sayozlashib qoldi.",
    "Amudaryo va Sirdaryo — Oʻrta Osiyoning ikki asosiy daryosi. Tarixan ikkala daryo ham Orol dengiziga quyilgan.",
    "Normal atmosfera bosimida suv 100 daraja Selsiyda qaynaydi. Togʻlarda bosim pastroq, shuning uchun suv pastroq haroratda qaynaydi.",
    "Oʻsimliklar energiyani quyosh nuridan oladi. Bu jarayon fotosintez deb ataladi va unda kislorod ajralib chiqadi.",
    "Oy — Yerning tabiiy yoʻldoshi. Oydan tushgan yorugʻlik Yerga bir soniyadan sal koʻproq vaqtda yetib keladi.",
    "Quyosh nuri Yerga taxminan 8 daqiqada yetib keladi. Quyosh — bizga eng yaqin yulduz.",
    "Oʻzbek tili turkiy tillar oilasiga kiradi. 1993-yildan boshlab u uchun lotin alifbosi rasman qabul qilingan.",
    "Rus tili — hind-yevropa oilasiga mansub sharqiy slavyan tili. U kirill alifbosida yoziladi.",
    "Toshkentni Samarqand va Buxoro bilan bogʻlaydigan tezyurar poyezd «Afrosiyob» deb ataladi. Toshkentdan Samarqandgacha yoʻl taxminan ikki soat davom etadi.",
    "Buxoroning tarixiy markazi YUNESKO Butunjahon merosi roʻyxatiga kiritilgan. Minorai Kalon XII asrda qurilgan.",
    "Xiva Xorazm viloyatida joylashgan. Uning ichki shahri Ichan qalʼa deb ataladi va baland devor bilan oʻralgan.",
    "Kompyuter maʼlumotlarni ikkilik kodda, yaʼni nol va birlar yordamida saqlaydi.",
]

# test savollari: (savol, toʻgʻri parcha raqami, javob)
questions = [
    ("Oʻzbekiston poytaxtida qancha odam yashaydi?", 0, "uch million"),
    ("Alisher Navoiy qayerda tugʻilgan?", 3, "Hirot"),
    ("Fotosintezda qaysi gaz ajralib chiqadi?", 9, "kislorod"),
    ("Quyosh nuri Yerga necha daqiqada yetib keladi?", 11, "8"),
    ("Suv necha darajada qaynaydi?", 8, "100"),
    ("Avitsenna qaysi yilda tugʻilgan?", 5, "980"),
    ("Toshkent va Samarqand oʻrtasidagi tezyurar poyezd qanday ataladi?", 14, "Afrosiyob"),
    ("Samarqanddagi rasadxonani kim qurdirgan?", 2, "Ulugʻbek"),
    ("Xivaning ichki shahri qanday ataladi?", 16, "Ichan qalʼa"),
    ("Oʻzbek tili uchun qaysi alifbo ishlatiladi?", 12, "lotin"),
    ("«Tib qonunlari» asarining muallifi kim?", 5, "Ibn Sino"),
    ("Al-Xorazmiy qaysi asrda yashagan?", 4, "IX"),
    ("Orol dengizi nega qurib qoldi?", 6, "sugʻorish"),
]

# zich enkoder uchun oʻquv juftliklari (savol, parcha): har bir parchaga 2 tadan, test savollaridan farq qiladi
train_questions = [
    ("Oʻzbekistonning eng katta shahri qaysi?", 0), ("Toshkentda qancha aholi bor?", 0),
    ("Samarqand markazida nima bor?", 1), ("Registon maydonida nechta madrasa bor?", 1),
    ("Ulugʻbek jadvalida nechta yulduz bor?", 2), ("Samarqandda rasadxona qachon paydo boʻlgan?", 2),
    ("Alisher Navoiy kim boʻlgan?", 3), ("Navoiy qaysi asrda yashagan?", 3),
    ("Al-Xorazmiy qayerdan chiqqan?", 4), ("Algoritm soʻzi kimning nomidan kelib chiqqan?", 4),
    ("Avitsenna qayerda tugʻilgan?", 5), ("Ibn Sino tibbiyot boʻyicha qanday kitob yozgan?", 5),
    ("Orol dengizi nega sayozlashdi?", 6), ("Orol dengizining suvi qayoqqa ketdi?", 6),
    ("Oʻrta Osiyoning asosiy daryolari qaysilar?", 7), ("Orol dengiziga qaysi daryolar quyilgan?", 7),
    ("Nega togʻlarda suv tezroq qaynaydi?", 8), ("Suvning qaynash harorati nimaga bogʻliq?", 8),
    ("Oʻsimliklar energiyani qayerdan oladi?", 9), ("Fotosintez nima?", 9),
    ("Oy nima?", 10), ("Oydan Yergacha yorugʻlik qancha vaqtda keladi?", 10),
    ("Yerga eng yaqin yulduz qaysi?", 11), ("Quyosh nuri qancha vaqt yoʻl bosadi?", 11),
    ("Oʻzbek tili qaysi oilaga kiradi?", 12), ("Oʻzbeklar qachondan lotin yozuvida yozadi?", 12),
    ("Rus tili qaysi alifboda yoziladi?", 13), ("Rus tili qaysi guruhga kiradi?", 13),
    ("Toshkentdan Samarqandga poyezdda necha soat yuriladi?", 14), ("Buxoroga qaysi poyezd boradi?", 14),
    ("Minorai Kalon qachon qurilgan?", 15), ("Qaysi shahar YUNESKO roʻyxatiga kiritilgan?", 15),
    ("Xiva qayerda joylashgan?", 16), ("Ichan qalʼani nima oʻrab turadi?", 16),
    ("Kompyuter maʼlumotni qanday saqlaydi?", 17), ("Ikkilik kod nima?", 17),
]
print(len(passages), "ta parcha,", len(questions), "ta test savoli,", len(train_questions), "ta oʻquv juftligi")

Qidiruv uchun matnni terminlar roʻyxatiga aylantiramiz: kichik harf, toʻxtash soʻzlarisiz (jumladan «kim», «qayerda», «qancha» kabi soʻroq soʻzlarisiz). Ikkinchi variant — qoʻshimchalarni kesish, shunda «Samarqanddagi» va «Samarqandda» bir xil oʻzakka keladi. Oʻzbek tili uchun tayyor stemmerlar kam, shuning uchun qoidalarni oʻzimiz yozamiz.

In [ ]:
import re
import numpy as np

TOKEN_RE = r"\d+|[^\W\d_]+(?:-[^\W\d_]+)*"     # ʻ (U+02BB) va ʼ (U+02BC) harf hisoblanadi, soʻz ichida qoladi
STOP = set("""va bu u uning unda uni bilan uchun ham deb yaʼni edi esa qaysi qanday qancha necha nechta
kim kimning nima nimaga nega qayerda qayerdan qachon qayoqqa
ataladi""".split())

# Oʻzbek tili uchun nltk da stemmer yoʻq, shuning uchun oddiy qoʻshimcha kesuvchini oʻzimiz yozamiz.
# Tartib oʻngdan chapga: kelishik -> egalik -> koʻplik (morfotaktika teskarisi).
CASES = ["ning", "dagi", "gacha", "dan", "da", "ga", "ka", "qa", "ni"]
POSS = ["imiz", "ingiz", "miz", "ngiz", "im", "ing", "si", "i"]
PLURAL = ["lar"]

def tokenize(text):
    return re.findall(TOKEN_RE, text)

def norm(w):
    return w.lower()

def stem(w):
    w = norm(w)
    for group in (CASES, POSS, PLURAL):
        for suf in sorted(group, key=len, reverse=True):
            if w.endswith(suf) and len(w) - len(suf) >= 3:
                w = w[: -len(suf)]
                break                     # har bir guruhdan ko'pi bilan bitta qo'shimcha
    return w

def analyze_words(text):
    # kichik harfdagi soʻzlar, toʻxtash soʻzlarisiz
    return [norm(w) for w in tokenize(text) if norm(w) not in STOP]

def analyze(text):
    # xuddi shu, lekin har bir soʻzdan qoʻshimchalar kesilgan
    return [stem(w) for w in tokenize(text) if norm(w) not in STOP]

for w in ["Samarqanddagi", "poytaxtida", "dengiziga", "Togʻlarda", "alifbosi", "yulduzlar"]:
    print(f"{w:<15} -> {stem(w)}")
print(analyze_words(passages[2]))
print(analyze(passages[2]))
print(analyze(questions[7][0]))

## 2. Siyrak qidiruv: BM25 ni noldan yozamiz

BM25 savol $q$ ning parcha $d$ da uchragan terminlari hissalarini qoʻshadi:

$$\text{BM25}(q,d)=\sum_{t\in q}\text{idf}(t)\cdot\frac{tf(t,d)\,(k_1+1)}{tf(t,d)+k_1\left(1-b+b\frac{|d|}{\text{avgdl}}\right)},\qquad \text{idf}(t)=\ln\left(1+\frac{N-df(t)+0.5}{df(t)+0.5}\right)$$

Kam uchraydigan termin (kichik $df$) katta vazn oladi. Terminning takrorlanishi yordam beradi, lekin toʻyinish bilan ($k_1$ parametri). Uzun parchalar jarimalanadi ($b$ parametri). Quyida savolning har bir termini eng yaxshi uchta parchaga qanday hissa qoʻshgani koʻrsatilgan.

In [ ]:
import math
from collections import Counter

class BM25:
    def __init__(self, texts, analyzer, k1=1.5, b=0.75):
        self.analyzer, self.k1, self.b = analyzer, k1, b
        self.docs = [Counter(analyzer(t)) for t in texts]          # har bir parchadagi terminlar chastotasi
        self.doc_len = np.array([sum(d.values()) for d in self.docs])
        self.avgdl = self.doc_len.mean()
        self.N = len(texts)
        self.df = Counter(t for d in self.docs for t in d)          # termin nechta parchada uchraydi

    def idf(self, t):
        return math.log(1 + (self.N - self.df[t] + 0.5) / (self.df[t] + 0.5))

    def term_score(self, t, i):
        tf = self.docs[i][t]
        norm_len = 1 - self.b + self.b * self.doc_len[i] / self.avgdl
        return self.idf(t) * tf * (self.k1 + 1) / (tf + self.k1 * norm_len)

    def scores(self, query):
        terms = self.analyzer(query)
        return np.array([sum(self.term_score(t, i) for t in terms) for i in range(self.N)])

bm25 = BM25(passages, analyze)
q = questions[7][0]
s = bm25.scores(q)
top = np.argsort(-s, kind="stable")[:3]
print(q)
print(f"{'termin':<14}{'df':>4}{'idf':>7}" + "".join(f"{'p' + str(i):>9}" for i in top))
for t in dict.fromkeys(analyze(q)):
    row = "".join(f"{bm25.term_score(t, i):9.2f}" for i in top)
    print(f"{t:<14}{bm25.df[t]:>4}{bm25.idf(t):7.2f}{row}")
print(f"{'BM25 yigʻindisi':<25}" + "".join(f"{s[i]:9.2f}" for i in top))
print()
for i in top:
    print(f"p{i} ({s[i]:.2f}): {passages[i]}")

### Qidiruv metrikalari

Har bir savol uchun toʻgʻri parchaning natijalar roʻyxatidagi **oʻrnini** (rank) topamiz (1 — birinchi). **Recall@k** — toʻgʻri parcha birinchi $k$ tasiga kirgan savollar ulushi. **MRR** (mean reciprocal rank) — $1/\text{oʻrin}$ ning oʻrtachasi: har doim birinchi boʻlsa 1.0, har doim ikkinchi boʻlsa 0.5.


### scikit-learn da xuddi shu: TF-IDF va kosinus

`TfidfVectorizer` har bir parcha uchun TF-IDF vektorini quradi va uni 1 uzunlikka normallaydi. Shunda savol vektori bilan parcha vektorining skalyar koʻpaytmasi kosinusga teng boʻladi. Solishtirish halol boʻlishi uchun oʻzimizning analizatorni beramiz.

In [ ]:
gold = np.array([g for _, g, _ in questions])

def ranks_from_scores(S, gold):
    # S: baholar matritsasi (savollar × parchalar)
    ranks = []
    for s, g in zip(S, gold):
        order = np.argsort(-s, kind="stable")
        ranks.append(int(np.where(order == g)[0][0]) + 1)
    return np.array(ranks)

def metrics(ranks):
    return {"R@1": np.mean(ranks <= 1), "R@3": np.mean(ranks <= 3), "MRR": np.mean(1 / ranks)}

def fmt(m):
    return "  ".join(f"{k}={v:.2f}" for k, v in m.items())

results = {}
for name, an in [("soʻzlar", analyze_words), ("oʻzaklar", analyze)]:
    model = BM25(passages, an)
    S = np.array([model.scores(q) for q, _, _ in questions])
    r = ranks_from_scores(S, gold)
    results["BM25, " + name] = r
    print(f"BM25, {name:<14} oʻrinlar: {r}   {fmt(metrics(r))}")


from sklearn.feature_extraction.text import TfidfVectorizer

for name, an in [("soʻzlar", analyze_words), ("oʻzaklar", analyze)]:
    vec = TfidfVectorizer(analyzer=an)            # BM25 dagi analizatorning oʻzi
    P = vec.fit_transform(passages)               # qatorlar normallangan: vektor uzunligi = 1
    Qm = vec.transform([q for q, _, _ in questions])
    S = (Qm @ P.T).toarray()                      # savol × parcha kosinuslari
    r = ranks_from_scores(S, gold)
    results["TF-IDF, " + name] = r
    print(f"TF-IDF, {name:<13} oʻrinlar: {r}   {fmt(metrics(r))}")

import matplotlib.pyplot as plt
names = list(results)
vals = np.array([[metrics(results[n])[k] for k in ["R@1", "R@3", "MRR"]] for n in names])
fig, ax = plt.subplots(figsize=(8, 3))
for j, k in enumerate(["R@1", "R@3", "MRR"]):
    ax.bar(np.arange(len(names)) + 0.25 * (j - 1), vals[:, j], width=0.25, label=k)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, fontsize=8)
ax.set_ylim(0, 1.05)
ax.set_title("Siyrak qidiruv: test savollaridagi sifat")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 3. Zich qidiruv: ikki enkoderli model va in-batch negatives

DPR (Dense Passage Retrieval) da savol va parcha **ikki xil** enkoder $E_Q$ va $E_P$ bilan bir xil oʻlchamli vektorlarga aylantiriladi, oʻxshashlik — skalyar koʻpaytma: $\text{sim}(q,p)=E_Q(q)\cdot E_P(p)$. Barcha parchalarning vektorlari oldindan hisoblanadi, shuning uchun qidiruv tez.

Oʻqitish: batchda $B$ ta juftlik (savol$_i$, parcha$_i$). $S=QP^\top$ matritsa $B\times B$ oʻlchamli: diagonalda musbat juftliklar, diagonaldan tashqarida boshqa savollarning parchalari — ular manfiy misol boʻlib xizmat qiladi (**in-batch negatives**). Yoʻqotish funksiyasi — qatorlar boʻyicha kross-entropiya:

$$L=-\frac{1}{B}\sum_i \log\frac{e^{S_{ii}}}{\sum_j e^{S_{ij}}}$$

BERT oʻrniga bizning enkoder — chiziqli proyeksiya: belgilar (belgi n-grammalari boʻyicha TF-IDF, 2–4 belgi) $D\times 64$ oʻlchamli $W_Q$ yoki $W_P$ matritsaga koʻpaytiriladi. Ikkala matritsani bir xil tasodifiy matritsadan boshlaymiz. Tasodifiy proyeksiya skalyar koʻpaytmalarni taxminan saqlaydi, shuning uchun oʻqitishdan oldin model oddiy belgi TF-IDF ga oʻxshaydi.

In [ ]:
# belgilar: belgi n-grammalari boʻyicha TF-IDF (soʻz ichida 2–4 belgi)
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True)
char_vec.fit(passages + [q for q, _ in train_questions])
X_pass = char_vec.transform(passages).toarray()
X_train = char_vec.transform([q for q, _ in train_questions]).toarray()
y_train = np.array([p for _, p in train_questions])
X_test = char_vec.transform([q for q, _, _ in questions]).toarray()
D = X_pass.shape[1]
print("belgilar soni D:", D)

def batch_loss(Wq, Wp, Xq, Xp):
    # Xq, Xp: B ta savol va ularning B ta parchasi belgilari (i-qator ↔ i-qator)
    Q, P = Xq @ Wq, Xp @ Wp            # chiziqli enkoderlar
    S = Q @ P.T                         # B×B: batchdagi barcha savollar × barcha parchalar
    A = np.exp(S - S.max(axis=1, keepdims=True))
    A /= A.sum(axis=1, keepdims=True)   # softmax qatorlar boʻyicha
    B = len(S)
    loss = -np.mean(np.log(np.diag(A)))
    dS = (A - np.eye(B)) / B            # dL/dS
    return loss, Xq.T @ (dS @ P), Xp.T @ (dS.T @ Q), S, A

rng = np.random.default_rng(0)
d = 64
W0 = rng.normal(0, 1 / np.sqrt(d), size=(D, d))    # tasodifiy proyeksiya D -> 64
Wq, Wp = W0.copy(), W0.copy()

demo_p = np.array([0, 3, 5, 9, 11, 14])                       # namoyish batchi: 6 ta parcha va har biriga bitta oʻquv savoli
demo_q = np.array([np.where(y_train == p)[0][0] for p in demo_p])
loss, gq, gp, S, A = batch_loss(Wq, Wp, X_train[demo_q], X_pass[demo_p])
np.set_printoptions(precision=2, suppress=True)
for qi in demo_q:
    print("  q:", train_questions[qi][0])
print("oʻxshashlik matritsasi S (qatorlar — savollar, ustunlar — parchalar):\n", S)
print("toʻgʻri parcha ehtimoli (softmax diagonali):", np.diag(A))
print(f"loss = {loss:.3f}   (tasodifiy taxmin: ln {len(demo_p)} = {np.log(len(demo_p)):.3f})")

# gradientni tekshirish: analitik va sonli (chekli ayirma)
j = int(np.nonzero(X_train[demo_q[0]])[0][0])
eps = 1e-5
Wq2 = Wq.copy(); Wq2[j, 0] += eps
numeric = (batch_loss(Wq2, Wp, X_train[demo_q], X_pass[demo_p])[0] - loss) / eps
print(f"Wq[j, 0] boʻyicha gradient: analitik {gq[j, 0]:.6f}, sonli {numeric:.6f}")

Gradient tushish bilan oʻrgatamiz: har qadamda 8 ta **turli** parcha va har biriga bitta oʻquv savoli olinadi. Agar batchdagi ikki savol bitta parchaga tegishli boʻlsa, «manfiy» misol aslida toʻgʻri boʻlib qoladi. Loss ni va oʻsha 6×6 matritsani oʻqitishdan oldin va keyin koʻramiz: diagonal yorqin boʻlishi kerak.

In [ ]:
def dense_scores(Xq, Wq, Wp):
    return (Xq @ Wq) @ (X_pass @ Wp).T

r_char = ranks_from_scores(X_test @ X_pass.T, gold)
r_before = ranks_from_scores(dense_scores(X_test, Wq, Wp), gold)
print("char TF-IDF (proyeksiyasiz):  ", fmt(metrics(r_char)))
print("zich model oʻqitishdan oldin:   ", fmt(metrics(r_before)))

S_before = batch_loss(Wq, Wp, X_train[demo_q], X_pass[demo_p])[3]
B, lr, steps = 8, 0.3, 300
history = []
for step in range(steps + 1):
    pids = rng.choice(len(passages), size=B, replace=False)        # 8 ta turli parcha: «soxta manfiy» misollarsiz
    qids = [rng.choice(np.where(y_train == p)[0]) for p in pids]
    loss, gq, gp, _, _ = batch_loss(Wq, Wp, X_train[qids], X_pass[pids])
    Wq -= lr * gq
    Wp -= lr * gp
    history.append(loss)
    if step % 50 == 0:
        print(f"qadam {step:3d}   loss = {loss:.3f}")
S_after = batch_loss(Wq, Wp, X_train[demo_q], X_pass[demo_p])[3]

r_dense = ranks_from_scores(dense_scores(X_test, Wq, Wp), gold)
results["zich (dual encoder)"] = r_dense
print()
print("oʻqitishdan keyin, oʻquv savollari:", fmt(metrics(ranks_from_scores(dense_scores(X_train, Wq, Wp), y_train))))
print("oʻqitishdan keyin, test savollari: ", fmt(metrics(r_dense)))

fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
ax[0].plot(history, lw=0.8)
ax[0].set_title("qadamlar boʻyicha loss")
ax[0].set_xlabel("qadam")
labels = ["p" + str(p) for p in demo_p]
for a, M, t in [(ax[1], S_before, "S oʻqitishdan oldin"), (ax[2], S_after, "S oʻqitishdan keyin")]:
    a.imshow(M, cmap="viridis")
    a.set_xticks(range(len(labels))); a.set_xticklabels(labels)
    a.set_yticks(range(len(labels))); a.set_yticklabels(["q" + str(p) for p in demo_p])
    a.set_title(t)
plt.tight_layout()
plt.show()

### Qaysi usul qayerda yutadi

Har bir savol uchun toʻgʻri parcha oʻrnini solishtiramiz. Zich model parafrazlarda oʻrgangan: masalan, oʻquv maʼlumotlarida «Avitsenna qayerda tugʻilgan?» → Ibn Sino haqidagi parcha bor edi. Parchaning oʻzida «Avitsenna» soʻzi yoʻq, BM25 uni topa olmaydi. Lekin BM25 savolda bor boʻlsa, kam uchraydigan soʻzlar va sonlarni aniq ushlaydi. Shuning uchun amalda koʻpincha **gibrid** ishlatiladi: ikkala usulning normallangan baholari yigʻindisi.

Eʼtibor bering: test savollarida oʻrgatilgan model dastlabki belgi TF-IDF dan albatta yaxshiroq boʻlmaydi. 36 ta oʻquv juftligi juda kam, model ularni qisman yodlab oladi (oʻquv savollarida MRR = 1.00). 13 ta savol boʻyicha «gʻolib» haqida xulosa chiqarmang: bitta savol farqi — bu shovqin. Haqiqiy DPR oʻn minglab juftliklarda oʻrgatiladi va BM25 natijalaridan «qiyin» manfiy misollar qoʻshiladi.

In [ ]:
r_bm25 = results["BM25, oʻzaklar"]
print(f"{'BM25':>5} {'zich':>7}  savol")
for (q, g, _), a, b in zip(questions, r_bm25, r_dense):
    mark = "  <- BM25" if a < b else ("  <- zich" if b < a else "")
    print(f"{a:>5} {b:>7}  {q}{mark}")

# gibrid: har bir savol boʻyicha normallangan baholar yigʻindisi (z-baholar)
S_bm = np.array([bm25.scores(q) for q, _, _ in questions])
S_de = dense_scores(X_test, Wq, Wp)
z = lambda M: (M - M.mean(axis=1, keepdims=True)) / (M.std(axis=1, keepdims=True) + 1e-9)
results["gibrid"] = ranks_from_scores(z(S_bm) + z(S_de), gold)
print()
for name in ["BM25, oʻzaklar", "TF-IDF, oʻzaklar", "zich (dual encoder)", "gibrid"]:
    print(f"{name:<22} {fmt(metrics(results[name]))}")

### Haqiqiy modelda

`intfloat/multilingual-e5-small` — koʻp tilli gap enkoderi (118 mln parametr, vektor oʻlchami 384, vaznlar fayli ≈470 MB). Savollar va parchalar uchun bitta model, lekin ular prefikslar bilan farqlanadi: savolga `"query: "`, parchaga `"passage: "` qoʻshiladi. Prefikslarsiz sifat sezilarli tushadi. Model oʻqitilgan maʼlumotlarda oʻzbekcha matnlar rus tilidagidan ancha kam, shuning uchun natija yomonroq boʻlishi mumkin. Katak faqat internetli Colab da ishlaydi.

In [ ]:
# Colab da sentence-transformers allaqachon oʻrnatilgan (aks holda: !pip install -q sentence-transformers)
try:
    from sentence_transformers import SentenceTransformer
    e5 = SentenceTransformer("intfloat/multilingual-e5-small")      # ≈470 MB, bir marta yuklanadi
    # E5 modellari uchun prefikslar majburiy
    P_e5 = e5.encode(["passage: " + p for p in passages], normalize_embeddings=True)
    Q_e5 = e5.encode(["query: " + q for q, _, _ in questions], normalize_embeddings=True)
    print("parcha vektorlari:", P_e5.shape)
    r_e5 = ranks_from_scores(Q_e5 @ P_e5.T, gold)
    print("multilingual-e5-small:", fmt(metrics(r_e5)))
    print("BM25 (oʻzaklar):    ", fmt(metrics(results["BM25, oʻzaklar"])))
    for (q, g, _), r in zip(questions, r_e5):
        print(f"  {r:>2}  {q}")
except Exception as e:
    print("Modelni yuklab boʻlmadi (Colab da internet kerak):", e)

## 4. Oʻquvchi (reader): parchadan javobni ajratamiz

Neyron oʻquvchi (DPR dagi BERT) har bir token uchun javob boshi va oxiri ehtimolini bashorat qiladi. Biz xuddi shu gʻoyalarni koʻrsatadigan oddiy qoidaviy oʻquvchi yozamiz:

1. **nomzodlar** — parchadagi savolda yoʻq soʻzlar (javob odatda savolni takrorlamaydi), toʻxtash soʻzlaridan tashqari;
2. **yaqinlik**: nomzod parchada $j$ oʻrinda topilgan har bir savol soʻzi $w_j$ boʻyicha $\sum_j \text{idf}(w_j)/|i-j|$ ball oladi;
3. **javob turi** soʻroq soʻzi boʻyicha: «qancha», «qaysi yilda» → son; «qayerda» → joy (oʻrin-payt yoki chiqish kelishigidagi bosh harfli soʻz); «kim» → ism (bosh harfli soʻz). Mos turdagi nomzodlar boʻlsa, faqat ulardan tanlaymiz;
4. **fokus**: «qaysi alifbo» savolida javob odatda «alifbo» soʻzidan bevosita oldin turadi («lotin alifbosi»), «… qanday ataladi?» savolida esa «deb ataladi» dan oldin («Afrosiyob» deb ataladi). Bunday nomzod +3 ball oladi.

Javob — eng yaxshi nomzod, shu turdagi qoʻshni nomzodlar bilan kengaytiriladi (3 soʻzgacha: «Ibn Sino», «uch million»). Bu javob boshi va oxiri ehtimollarining qoʻpol oʻrinbosari.

In [ ]:
SPAN_RE = r"\d+|[^\W\d_]+(?:-[^\W\d_]+)*|[^\w\s]"     # TOKEN_RE kabi, tinish belgilari alohida token
# kutilayotgan javob turi soʻroq soʻzidan aniqlanadi
TYPE_PATTERNS = [
    ("number", r"qancha|necha|qaysi yil|qaysi asr"),
    ("place",  r"qayerda|qayerdan"),
    ("person", r"\bkim\b"),
    ("name",   r"qanday ataladi"),
]
NUMBER_RE = r"\d+|bir|ikki|uch|toʻrt|besh|oʻn|yuz|ming|million"

def is_place(toks, i):
    # joy: oʻrin-payt yoki chiqish kelishigidagi atoqli ot («Hirotda», «Xorazmdan») yoki «Buxoro yaqinida»
    t = toks[i].lower()
    nxt = toks[i + 1].lower() if i + 1 < len(toks) else ""
    return t.endswith(("da", "dan", "dagi")) or nxt in {"yaqinida", "atrofida"}

def question_focus(question):
    # «qaysi alifbo» -> («alifbo», +1): javob «alifbo» soʻzidan bevosita OLDIN turadi («lotin alifbosi»)
    # «qanday ataladi» -> («deb», +1): javob «deb ataladi» dan oldin turadi («Afrosiyob» deb ataladi)
    ql = question.lower()
    if "qanday ataladi" in ql:
        return ("deb", +1)
    m = re.search(r"\bqaysi\s+(\w+)", ql)
    return (stem(m.group(1)), +1) if m else None

def is_cap(tok):
    return tok[:1].isupper()

def answer_type(question):
    ql = question.lower()
    for typ, pattern in TYPE_PATTERNS:
        if re.search(pattern, ql):
            return typ
    return None

def token_type_ok(toks, i, typ):
    t = toks[i]
    if typ == "number":
        return bool(re.fullmatch(NUMBER_RE, t.lower()) or re.fullmatch(r"[IVXLC]+", t))
    if typ == "name":
        return True                  # nom istalgan soʻz boʻlishi mumkin
    if typ == "place":
        return is_cap(t) and is_place(toks, i)
    if typ == "person":
        return is_cap(t) and not is_place(toks, i)
    return True

def next_word(toks, i, step):
    # nearest word token to the left (step=-1) or right (step=+1), punctuation skipped
    k = i + step
    while 0 <= k < len(toks) and not toks[k][0].isalnum():
        k += step
    return k if 0 <= k < len(toks) else None

def read(question, passage, bm=bm25, verbose=False):
    toks = re.findall(SPAN_RE, passage)
    q_stems = set(analyze(question))
    typ = answer_type(question)
    focus = question_focus(question)          # (oʻzak, tomon) yoki None
    match_pos = [j for j, t in enumerate(toks) if t[0].isalnum() and stem(t) in q_stems]
    # nomzodlar: parchadagi savolda yoʻq soʻzlar, toʻxtash soʻzlarisiz
    cand = [i for i, t in enumerate(toks)
            if t[0].isalnum() and norm(t) not in STOP and stem(t) not in q_stems]
    score = {}
    for i in cand:
        # parchada topilgan savol soʻzlariga yaqinlik, idf vazni bilan
        prox = sum(bm.idf(stem(toks[j])) / abs(i - j) for j in match_pos)
        # savolning fokus soʻzidan bevosita oldin turgan nomzod
        k = next_word(toks, i, focus[1]) if focus else None
        near_focus = k is not None and stem(toks[k]) == focus[0]
        score[i] = prox + (3.0 if near_focus else 0.0)
    typed = [i for i in cand if typ and token_type_ok(toks, i, typ)]
    pool = typed if typed else cand                  # kerakli turdagi nomzodlar boʻlsa — faqat ulardan tanlaymiz
    best = max(pool, key=lambda i: score[i])
    # javobni shu turdagi qoʻshni nomzodlar bilan kengaytiramiz (3 soʻzgacha)
    span = [best]
    if typ is not None:
        for step in (-1, 1):
            k = best + step
            while k in score and k in pool and len(span) < 3:
                span.append(k); k += step
    span.sort()
    answer = " ".join(toks[i] for i in span)
    if verbose:
        print(f"javob turi: {typ}, fokus: {focus}, parchadagi savol soʻzlari: {[toks[j] for j in match_pos]}")
        for i in sorted(pool, key=lambda i: -score[i])[:4]:
            print(f"    {toks[i]:<14} {score[i]:5.2f}")
    return answer, toks, score, span

for k in [1, 9, 10]:
    q, g, ans = questions[k]
    print(q)
    pred = read(q, passages[g], verbose=True)[0]
    print("  ->", pred, "  (toʻgʻrisi:", ans + ")\n")

Bitta parchadagi nomzodlar ballari (kulrang — savol soʻzlari va toʻxtash soʻzlari, toʻq sariq — tanlangan javob):

In [ ]:
q, g, ans = questions[1]
pred, toks, score, span = read(q, passages[g])
x = np.arange(len(toks))
h = [score.get(i, 0) for i in x]
colors = ["#D9822B" if i in span else ("#2B55C9" if i in score else "#BBBBBB") for i in x]
fig, ax = plt.subplots(figsize=(11, 3))
ax.bar(x, h, color=colors)
ax.set_xticks(x)
ax.set_xticklabels(toks, rotation=70, fontsize=8)
ax.set_title("nomzodlar ballari: " + q)
plt.tight_layout()
plt.show()

### Butun tizim: BM25 → oʻquvchi

BM25 natijalaridan birinchi parchani olib, oʻquvchini ishga tushiramiz. Topilgan qism toʻgʻri javobni oʻz ichiga olsa yoki uning ichida boʻlsa, javob toʻgʻri hisoblanadi. Oʻzbek tilida javob koʻpincha qoʻshimcha bilan chiqadi («Hirotda»): token va morfema chegarasi mos kelmaydi. Xatolar qayerda qidiruvda, qayerda oʻquvchida ekanini koʻring. «Nega…» savoli sabab-javobni (butun gapni) talab qiladi, bizning oʻquvchi esa faqat qisqa qismlarni ajrata oladi.

In [ ]:
# BM25 natijalaridagi birinchi parcha -> oʻquvchi
correct = 0
for q, g, ans in questions:
    top1 = int(np.argmax(bm25.scores(q)))
    pred = read(q, passages[top1])[0]
    ok = ans.lower() in pred.lower() or pred.lower() in ans.lower()
    correct += ok
    print(f"{'+' if ok else '-'} p{top1:<3} {pred:<22} ({ans})  {q}")
print(f"\ntoʻgʻri javoblar: {correct} / {len(questions)}")

## 5. Topshiriqlar

1. **BM25 parametrlari.** $k_1 \in \{0.5, 1.2, 2.0\}$ va $b \in \{0, 0.75, 1\}$ uchun MRR ni hisoblang (9 ta kombinatsiya, jadval). $b=0$ nimani anglatadi? Nega bizning qisqa parchalarimizda farq kichik?
2. **Oʻz savollaringiz.** Parchalarga 4 ta yangi savol tuzing: 2 tasi parcha soʻzlarining sinonimlari bilan (parafraz), 2 tasi kam uchraydigan ism yoki son bilan. BM25 va zich model natijalaridagi oʻrinlarni solishtiring.
3. **Batch oʻlchami.** Ikki enkoderli modelni qaytadan (`W0` dan) $B = 2, 4, 8, 16$ bilan bir xil qadamlar sonida oʻrgating. Yakuniy loss va testdagi MRR qanday oʻzgaradi? Nega $B=2$ da masala juda oson?
4. **Vaznli gibrid.** Oddiy yigʻindi oʻrniga $\alpha\cdot z(\text{BM25})+(1-\alpha)\cdot z(\text{dense})$ dan foydalaning va 0, 0.1, …, 1 toʻrida eng yaxshi $\alpha$ ni toping. Nega $\alpha$ ni aynan test savollarida tanlash yomon amaliyot?
5. **Oʻquvchi.** `TYPE_PATTERNS` ga «qachon» savollari uchun tur qoʻshing (javob — 3–4 raqamli son) va uni «Ibn Sino qachon tugʻilgan?» va «Minorai Kalon qachon qurilgan?» savollarida tekshiring. Qoʻshimcha: «Hirotda» kabi javoblardan kelishik qoʻshimchasini `stem()` bilan olib tashlang. Qanday xatolar qoldi?

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing